In [4]:
from urllib.parse import urlparse

def clean_domain(domain):
    try:
        domain = domain.strip().lower()
        if domain.startswith("http"):
            from urllib.parse import urlparse
            domain = urlparse(domain).netloc
        return domain.split(':')[0]  # Remove port if present
    except:
        return None

In [5]:
# === WHOIS Lookup ===
def get_whois_info(domain):
    try:
        w = whois.whois(domain)
        return {
            'domain': domain,
            'creation_date': w.creation_date if isinstance(w.creation_date, datetime) else None,
            'registrar': w.registrar,
            'country': w.country,
            'city': getattr(w, 'city', None)
        }
    except:
        return {
            'domain': domain,
            'creation_date': None,
            'registrar': None,
            'country': None,
            'city': None
        }

In [6]:
import pandas as pd
import whois
from datetime import datetime
from concurrent.futures import as_completed, ThreadPoolExecutor
import time
import os

# ----------------s----------
# CONFIGURATION
# --------------------------
batch_size = 5000
batch_start = 0     # Change this if continuing from a later batch
max_workers = 5
source_file = "url_with_ip_and_domain_features.csv"
output_file = "whois_master.csv"  # Final merged file
log_file = "whois_progress.log"   # Tracks progress


# Load data
df = pd.read_csv(source_file)
df = df[df['ip'].notnull()]
df['domain'] = df['domain'].apply(clean_domain)
domains = df['domain'].dropna().unique().tolist()
print(df['domain'].head(10).tolist())

['www.dghjdgf.com', 'horizonsgallery.com', 'docs.google.com', 'www.henkdeinumboomkwekerij.nl', 'optimistic-pessimism.com', 'jameshowardmusic.com', 'xini.eu', 'horizonsgallery.com', 'docs.google.com', 'stthomasedu.ucoz.ua']


In [8]:
# Load previous progress
if os.path.exists(output_file):
    done_df = pd.read_csv(output_file)
    done_domains = set(done_df['domain'].tolist())
    print(f"✅ Already processed: {len(done_domains)} domains")
else:
    done_df = pd.DataFrame()
    done_domains = set()

remaining_domains = [d for d in domains if d not in done_domains]
print(f"🕒 Domains remaining: {len(remaining_domains)}")

# --------------------------
# Process in batches
# --------------------------
for batch_start in range(0, len(remaining_domains), batch_size):
    batch_end = min(batch_start + batch_size, len(remaining_domains))
    current_batch = remaining_domains[batch_start:batch_end]
    print(f"\n🚀 Processing batch {batch_start}-{batch_end} ...")

    start_time = time.time()
    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_domain = {executor.submit(get_whois_info, domain): domain for domain in current_batch}
        for future in as_completed(future_to_domain):
            result = future.result()
            results.append(result)

    batch_df = pd.DataFrame(results)
    
    # Append to master file
    if os.path.exists(output_file):
        batch_df.to_csv(output_file, mode='a', header=False, index=False)
    else:
        batch_df.to_csv(output_file, index=False)

    print(f"✅ Batch {batch_start}-{batch_end} done in {round((time.time() - start_time)/60, 2)} minutes")
    
    # Save progress
    with open(log_file, 'a') as f:
        f.write(f"Batch {batch_start}-{batch_end} completed\n")

✅ Already processed: 127302 domains
🕒 Domains remaining: 0


In [10]:
df_main = pd.read_csv("url_with_ip_and_domain_features.csv")
whois_df = pd.read_csv("whois_master.csv")

final_df = pd.merge(df_main, whois_df, on='domain', how='left')
final_df.to_csv("urls_full_with_ip_domain_and_whois.csv", index=False)